# Elexon exploratory checks

Phase 0 exploration for Wind Forecast Decision Desk. This notebook only uses the Elexon Insights API and deliberately requests the late-December publication window needed for January delivery decisions.

In [ ]:
import pandas as pd
import requests

API = 'https://data.elexon.co.uk/bmrs/api/v1'
WINDOW = {'from': '2025-12-30T00:00:00Z', 'to': '2026-02-01T00:00:00Z'}

def get_stream(dataset: str, **params):
    response = requests.get(f'{API}/datasets/{dataset}/stream', params=params, timeout=60)
    response.raise_for_status()
    return pd.DataFrame(response.json())


In [ ]:
# Inspect the exact returned schema before ingestion code is written.
windfor = get_stream('WINDFOR', **WINDOW)
fuelhh = get_stream('FUELHH', **WINDOW, fuelType='WIND')
display(windfor.head(), windfor.dtypes, fuelhh.head(), fuelhh.dtypes)
print('WINDFOR rows:', len(windfor), 'FUELHH rows:', len(fuelhh))

In [ ]:
# Confirm repeated forecasts per target and inspect decision-relevant horizons.
windfor['publishTime'] = pd.to_datetime(windfor['publishTime'], utc=True)
windfor['startTime'] = pd.to_datetime(windfor['startTime'], utc=True)
windfor['horizon_hours'] = (windfor['startTime'] - windfor['publishTime']).dt.total_seconds() / 3600
duplicates = windfor.groupby('startTime').size().describe()
print(duplicates)
windfor['horizon_hours'].describe(percentiles=[.05, .5, .95])

## Expected findings

- `WINDFOR` should have multiple `publishTime` values for the same `startTime`.
- A January-only target window is insufficient: decisions for 1 January require forecasts published on 31 December.
- The ingestion phase validates the returned field names and saves a reproducible snapshot; no analysis is based on this notebook's transient output.